In [1]:
from datasets import load_dataset
import pandas as pd

# load dataset
dataset = load_dataset("allenai/drug-combo-extraction")

# convert to dataframe
train_df = pd.DataFrame(dataset['train'])

print(train_df.shape)
print(train_df.columns.tolist())
print(train_df.head(3))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

final_train_set.jsonl: 0.00B [00:00, ?B/s]

final_test_set.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1362 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/272 [00:00<?, ? examples/s]

(1362, 6)
['doc_id', 'sentence', 'spans', 'rels', 'paragraph', 'source']
                             doc_id  \
0  26548f0a805861520e4097b0f1fb278f   
1  75080bf07d4ac4a75b2906ffb6f7e2f5   
2  401db5d57ea1f07ccd46d84bb8bb8049   

                                            sentence  \
0  Among the anti-herpesvirus agents , aciclovir ...   
1  The present study examined the efficacy of sin...   
2  He was initially treated with combination chem...   

                                               spans  \
0  [{'span_id': 0, 'text': 'aciclovir', 'start': ...   
1  [{'span_id': 0, 'text': 'D-cycloserine', 'star...   
2  [{'span_id': 0, 'text': 'vincristine', 'start'...   

                                                rels  \
0                                                 []   
1                                                 []   
2  [{'class': 'COMB', 'spans': [0, 1, 2], 'is_con...   

                                           paragraph source  
0  Antiviral drugs: current state

In [2]:
# keep only needed columns
train_df = train_df[['spans', 'rels']].copy()

print(train_df.shape)
print(train_df.head(3))

(1362, 2)
                                               spans  \
0  [{'span_id': 0, 'text': 'aciclovir', 'start': ...   
1  [{'span_id': 0, 'text': 'D-cycloserine', 'star...   
2  [{'span_id': 0, 'text': 'vincristine', 'start'...   

                                                rels  
0                                                 []  
1                                                 []  
2  [{'class': 'COMB', 'spans': [0, 1, 2], 'is_con...  


In [3]:
from itertools import combinations

def extract_data(row):
    spans = row['spans']
    rels = row['rels']

    # create dictionary mapping span_id to drug name
    id_to_name = {span['span_id']: span['text'] for span in spans}

    # get all drug ids in this sentence
    drug_ids = list(id_to_name.keys())

    # generate all possible drug pairs using combinations
    # combinations([0,1,2], 2) → [(0,1), (0,2), (1,2)]
    # this ensures we check every drug against every other drug
    pairs = list(combinations(drug_ids, 2))

    # get interacting pairs from rels
    # rels contains spans that interact together
    interacting_pairs = set()
    for rel in rels:
        # use combinations to get all pairs within each interaction group
        # example: spans=[0,1,2] → pairs (0,1), (0,2), (1,2)
        for i, j in combinations(rel['spans'], 2):
            interacting_pairs.add((min(i,j), max(i,j)))

    # build final rows
    # label 1 = interaction exists, 0 = no interaction
    result = []
    for id1, id2 in pairs:
        pair = (min(id1,id2), max(id1,id2))
        result.append({
            'drug_1': id_to_name[id1],
            'drug_2': id_to_name[id2],
            'interaction': 1 if pair in interacting_pairs else 0
        })
    return result

# apply to all rows in dataset
all_pairs = []
for _, row in train_df.iterrows():
    all_pairs.extend(extract_data(row))

# convert to dataframe
pairs_df = pd.DataFrame(all_pairs)

print(pairs_df.shape)
print(pairs_df.head(10))
print("\nlabel distribution:")
print(pairs_df['interaction'].value_counts())

(5466, 3)
         drug_1        drug_2  interaction
0     aciclovir  valaciclovir            0
1     aciclovir   penciclovir            0
2     aciclovir   famciclovir            0
3     aciclovir   idoxuridine            0
4     aciclovir  trifluridine            0
5     aciclovir   ganciclovir            0
6     aciclovir     foscarnet            0
7     aciclovir     cidofovir            0
8  valaciclovir   penciclovir            0
9  valaciclovir   famciclovir            0

label distribution:
interaction
0    3141
1    2325
Name: count, dtype: int64


In [4]:
# save to csv
pairs_df.to_csv('drug_combo_pairs.csv', index=False)

print("File saved successfully! ")
print("Shape:", pairs_df.shape)
print("\nLabel distribution:")
print(pairs_df['interaction'].value_counts())

File saved successfully! 
Shape: (5466, 3)

Label distribution:
interaction
0    3141
1    2325
Name: count, dtype: int64


In [6]:
from google.colab import files

# download the file
files.download('drug_combo_pairs.csv')

print("File downloaded! ")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

File downloaded! 
